# Modelo con LTSM, este modelo ya incorpora teching force

In [1]:
version = "5_1"

In [2]:
import pickle
import pandas as pd
import numpy as np
import json

pd.set_option("display.max_rows", None)   # Muestra todas las filas
pd.set_option("display.max_columns", None)  # Muestra todas las columnas
pd.set_option("display.width", None)     # No corta la tabla en varias líneas
pd.set_option("display.max_colwidth", None)  # Muestra el contenido de celdas completo

In [3]:
with open("../../data/normalized/df_normalized_dayOfYear.pk1", "rb") as f:
    df_data = pickle.load(f)

In [6]:
df_model = df_data.drop(columns=[
    "ride_id",
    "started_at"
])

In [7]:
# Variables de entrada
context = df_model.drop(columns=['start_station_idx'])
start = df_model['start_station_idx']
end   = df_model['end_station_idx']

In [8]:
context.shape

(9510782, 22)

In [9]:
start.shape

(9510782,)

In [10]:
end.shape

(9510782,)

In [11]:
from sklearn.model_selection import train_test_split

ctx_train, ctx__test, start_train, start_test, end_train, end_test = train_test_split(
    context, 
    start, 
    end, 
    test_size=0.2, 
    random_state=42
)

# ---------- Normalizar índices de estaciones con un offset común
station_offset = int(min(start.min(), end.min()))
# desplazamos todos para que el mínimo sea 0
# start_train_shift = (start_train - station_offset).astype(int)
# start_test_shift  = (start_test  - station_offset).astype(int)
# end_train_shift   = (end_train   - station_offset).astype(int)
# end_test_shift    = (end_test    - station_offset).astype(int)

num_stations = int(max(start.max(), end.max()) - station_offset + 1)
num_features = context.shape[1]

print(f"Number of stations: {num_stations}")
print(f"Number of features: {num_features}")
print(f"Station offset: {station_offset}")

# ---------- Preparar inputs en la forma que requieren los Embeddings: (n,1)
input_train_context = ctx_train.values.astype(np.float32)
input_test_context  = ctx__test.values.astype(np.float32)

input_train_start = start_train.values.reshape(-1, 1)
input_test_start  = start_test.values.reshape(-1, 1)

# Labels (etiquetas) para clasificación: estación destino (y_end)
output_train_end = end_train.values.astype(int)
output_test_end  =end_test.values.astype(int)

# ---------- split del train en train/val
input_train_context, input_val_context, input_train_start, input_val_start, output_train_end, output_val_end = train_test_split(
    input_train_context,
    input_train_start,
    output_train_end,
    test_size=0.2,
    random_state=42,
    shuffle=True # No se puede estratificar porque hay estaciones con muy pocos datos
)

Number of stations: 1912
Number of features: 22
Station offset: 0


In [14]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

embedding_dim = int(np.ceil(np.sqrt(num_stations)))
timesteps = 1  # ya que tu contexto no es secuencia real

print(f"Embedding dimension: {embedding_dim}")

# ---------------------------
# Input 1: contexto (vector)
# ---------------------------
context_input = layers.Input(shape=(num_features,), name="context")

# Convertimos el vector en una "secuencia" de longitud 1
x_ctx = layers.Reshape((1, num_features))(context_input)

# LSTM procesando el contexto
x_ctx = layers.LSTM(128, return_sequences=False)(x_ctx)
x_ctx = layers.Dropout(0.3)(x_ctx)

# ---------------------------
# Input 2: estación origen
# ---------------------------
start_input = layers.Input(shape=(1,), name="start_station_input")

start_emb = layers.Embedding(num_stations, embedding_dim)(start_input)
start_emb = layers.Flatten()(start_emb)
start_emb = layers.Dense(32, activation="relu")(start_emb)

# ---------------------------
# Concatenación
# ---------------------------

x = layers.Concatenate()([x_ctx, start_emb])

x = layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(64, activation="relu")(x)

end_output = layers.Dense(num_stations, activation="softmax")(x)

model = Model(inputs=[context_input, start_input], outputs=end_output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Embedding dimension: 44


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ context             │ (None, 22)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ start_station_input │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 22)     │          0 │ context[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 44)     │     84,128 │ start_station_in… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 128)       │     77,312 │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 44)        │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 32)        │      1,440 │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 160)       │          0 │ dropout_2[0][0],  │
│ (Concatenate)       │                   │            │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 256)       │     41,216 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 256)       │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │     32,896 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │      8,256 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 1912)      │    124,280 │ dense_7[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 369,528 (1.41 MB)

 Trainable params: 369,528 (1.41 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit(
    {"context": input_train_context, "start_station_input": input_train_start},
    output_train_end,
    validation_data=(
        {"context": input_val_context, "start_station_input": input_val_start},
        output_val_end
    ),
    epochs=50,
    batch_size=256,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )
    ]
)


Epoch 1/50
23777/23777 ━━━━━━━━━━━━━━━━━━━━ 186s 8ms/step - accuracy: 0.2204 - loss: 2.9906 - val_accuracy: 0.1539 - val_loss: 3.7839
Epoch 2/50
23777/23777 ━━━━━━━━━━━━━━━━━━━━ 219s 8ms/step - accuracy: 0.2723 - loss: 2.7060 - val_accuracy: 0.1572 - val_loss: 5.0331
Epoch 3/50
23777/23777 ━━━━━━━━━━━━━━━━━━━━ 211s 9ms/step - accuracy: 0.2778 - loss: 2.7026 - val_accuracy: 0.1521 - val_loss: 5.9008
Epoch 4/50
23777/23777 ━━━━━━━━━━━━━━━━━━━━ 210s 9ms/step - accuracy: 0.2806 - loss: 2.6950 - val_accuracy: 0.1436 - val_loss: 6.3319
Epoch 5/50
23777/23777 ━━━━━━━━━━━━━━━━━━━━ 211s 9ms/step - accuracy: 0.2833 - loss: 2.6829 - val_accuracy: 0.1199 - val_loss: 6.2258
Epoch 6/50
23777/23777 ━━━━━━━━━━━━━━━━━━━━ 211s 9ms/step - accuracy: 0.2797 - loss: 2.7123 - val_accuracy: 0.0936 - val_loss: 7.8994
